# Introduction

Similar to the last notebook, parts of this guide can be run via the cells included in jupyter. For any applications that require the TBrowser or similar separate windows, you will need to use a stand-alone terminal with ROOT setup.

A terminal can be accessed within this notebook via:
- File
    - New
        - Terminal 

Alternatively, you can open a terminal on your machine.

From the terminal use the command `root -l` to activate root. The output from the terminal should be similar to what is seen below:

The `-l` flag means we will run root without showing the splash screen. I would recommend doing this by default (and going so far as aliasing `root` to `root -l` in your `bashrc`). Why? Because this splash screen can cause issues if you're working remotely on say, Viking, and root is trying to open a pointless splash screen.

Run the cell below before continuing on:

In [ ]:
%jsroot on

# Trees

## Exploring a Tree

The main type of file used within high energy physics will be the TTree class. Each column within the Tree are instances of the TBranch class. These are the branches that store data. 

High energy physics often happens on an event by event basis such as when a particle hits a detector, all the data contained by that particle is read as a single event. The branches store these events within them.

The branches can contain any type of data such as:
- Fundamental types: int, float
- C++ standard collections: std::vector, std::map
- User created C++ classes

We will be using the same file for trees as mentioned in the previous section, `Day_1_Tutorial_Input.root`. 

Open this file in your terminal via:

`root -l Day_1_Tutorial_Input.root`

Then, open a new TBrowser with:

`new TBrowser()`

### Task - Explore the events tree

Within our file on the TBrowser, open our `events` tree. As you will see, this tree contains an enormous bnumber of collections (branches). Each collection corresponds to some specific set of information. This can be information corresponding to a specific detector (`B0ECalClusters` for example is a collection relating to the B0 EM calorimeter).

You can manually sort through these, but you can also use the filter (the funnel icon) to find branches/collections of interest. Wild cards, `*` can be used to help here.

Take a few minutes to explore the collections in this tree. Then:

- Find and open the `MCParticles` branch.
- Take a look at the information that is contained within.
- Try double clicking some branch elements, this should automatically make a histogram. Interpret the output and see if you can identify what each element corresponds to.

This branch corresponds to **Truth** information for our events. The tree we're looking at is the output of a simulation. As such, we retain our truth. We know what initially went into the event and what actually happened.

## Drawing From Trees Directly with TTree->Draw()

The code below opens this file and accesses the events tree contained within. It then draws a histogram from the data contained in the MCParticles.charge branch.

In [ ]:
TFile* f = new TFile("Day_1_Tutorial_Input.root", "READ");
TTree* tree = (TTree*)f->Get("events");
TCanvas * c1 = new TCanvas("c", "c", 600, 600);
tree->Draw("MCParticles.charge");
c1->Draw();

This shows data about the detected particles charge. As you can see the data is compact and not readable. Knowing that charge should be in whole number increments we can adjust the histogram to be more usable.

By default, when you draw a branch like this (or by double clicking in a TBrowser) ROOT will automatically pick a range to fit all values into the histogram, split over 100 bins for 1D histograms and 40x40 bins for a 2D histogram.

Usually, this makes a histogram we can vaguely interpret. **However**, we will quickly encounter cases where it does not.

Consider what would happen if your code is determining say, the mass of an isotope created in an event. In AMU, we excpect values in the range of ~1-300 or so here. But perhaps your code is set to assign events where no isotope is created a value of -1000000. In this case, your autogenerated histogram won't make sense. All of your values of interest will be squeezed into one bin such that we can fit the range -1000000 to 300.

But, we don't have to suffer ROOT's autobinning silently!

## Editing the histogram

We can change the range that is displayed to only show useful data from within the draw command as follows:

In [ ]:
TCanvas * c1 = new TCanvas("c", "c", 600, 600);

TH1F *h1 = new TH1F("h1", "MCParticles.charge;charge;events", 100, -5, 5);
tree->Draw("MCParticles.charge>>h1");
h1->Draw();

c1->Draw();

The code above creates a histogram referred to as h1, The histogram can be setup and changed on the fly. The data from the relevant tree branch is then used within the created histogram using the `tree->Draw("MCParticles.charge>>h1");` line

From here the histogram can be changed on the fly such as the range of the x-axis below

**Note: When using root in a terminal the canvas and redrawing step is obsolete, this only needs to be done within the notebook**

Note that in addition to providing a title for the histogram `MCParticles.charge` we also provided axis labels - `;charge;events` - the X and Y labels respectively. 

I would strongly advise making a habit of clearly labelling your axes **including units** whenever you make a histogram.

In [ ]:
h1->GetXaxis()->SetRangeUser(-1.5, 1.5);
c1->Draw();

Knowing details about what you are expecting from your data can help to set the correct range. Otherwise create the histogram from the raw data and edit the layout, range and number of bins to fit what has been obtained.

Note that we don't have to define the histogram ahead of time either, we can specify it in the `Draw` command directly:

In [ ]:
tree->Draw("MCParticles.charge>>h1(100, -5, 5)");
c1->Draw();

## Applying Conditions (Cuts)

Often is is useful to be able to draw histograms of some data while applying conditions to the particles based off of other data. For the example here we will be plotting the charges of particles that have a non-zero mass. 

In [ ]:
TCanvas * c1 = new TCanvas("c", "c", 600, 600);

TH1F *h1 = new TH1F("h1", "MCParticles.charge;charge;events", 100, -5, 5);
tree->Draw("MCParticles.charge>>h1", "MCParticles.mass > 0");
h1->Draw();
h1->GetXaxis()->SetRangeUser(-1.5, 1.5);

c1->Draw();

Note that we can stack our conditions/cuts in this manner too.

In [ ]:
TH1F *h1 = new TH1F("h1", "MCParticles.charge;charge;events", 100, -5, 5);
tree->Draw("MCParticles.charge>>h1", "MCParticles.mass > 0 && MCParticles.PDG != 11");
h1->Draw();
h1->GetXaxis()->SetRangeUser(-1.5, 1.5);

c1->Draw();

Here we requested particles with non-zero mass that **do not** have a PDG code of 11. We'll discuss PDG codes a bit more in a moment.

## Draw Options

We can also add draw options to our draw command too. We straightforwardly just add an extra argument at the end with our options. For example:

In [ ]:
TCanvas * c2 = new TCanvas("c2", "c2", 600, 600);
tree->Draw("MCParticles.charge>>h2(100, -5, 5)", "MCParticles.mass > 0 && MCParticles.PDG != 11", "HISTE");
c2->Draw();

Note that in our notebook, this doesn't quite render correctly (it should work directly from a terminal though).

## 2D Histograms

When used correctly, 2D histograms can be a very good way to spot correlations between variables in your data. As with 1D histograms, we can create and draw these using TTree draw. Let's do this two ways.

In [ ]:
TCanvas * c3 = new TCanvas("c3", "c3", 600, 600);

TH2F *h2D1 = new TH2F("h2D1", "MCParticles.charge vs MCParticles.mass;charge;mass", 200, -5, 5, 200, 0, 10);
tree->Draw("MCParticles.mass:MCParticles.charge>>h2D1");
h2D1->Draw("COLZ");

c3->Draw();

OR

In [ ]:
TCanvas * c4 = new TCanvas("c4", "c4", 600, 600);
tree->Draw("MCParticles.mass:MCParticles.charge>>h2D2(200, -5, 5, 200, 0, 10)","", "COLZ");
c4->Draw();

Note that we have used the `COLZ` draw option to draw our 2D histogram with the Z-axis as a colour scale. By default, this uses the kBIRD colour palette. This is an accessible palette and is generally recommended.

Note that as with the X and Y scales, ROOT will try to sensibly scale your Z-axis (and corresponding colour scale). **However**, you may want to adjust this yourself **OR** use a logarithmic Z-scale.

## More Complex Expressions

More complex expressions can be plotted from the data contained within a tree. This includes trig functions as well as log expressions.

Below is an example of plotting the log of the x momentum for our MC particles.

**Note, this doesn't have any specific meaning for data and is being used as an example of how to use the functions.**

In [ ]:
TCanvas * c1 = new TCanvas("c", "c", 600, 600);

/* note the log expression in the argument */
tree->Draw("log(MCParticles.momentum.x)");

c1->Draw();

## Task

Using what we've seen with TTree draw, we're going to use it to do a mini analysis of our MC particles branch.

Some brief extra info before we continue. There are two branches we're going to make extensive use of for this task:

- `MCParticles.PDG`
    - These are the PDG codes for our MCParticles. Each particle has a unique PDG code using a [scheme established in this document](https://pdg.lbl.gov/2019/reviews/rpp2018-rev-monte-carlo-numbering.pdf).
    - As this collection is **MC Truth** information, the PDG code tells us what a particle in the collection **actually** is.
    - Some common PDG codes include:
        - 11 - An electron
        - 22 - A photon
        - 2212 - A proton
        - 2112 - A neutron
        - 211 - A $\pi^{+}$
        - 321 - A $K^{+}$
    - Note, to get an anti-particle, just use a -ve sign. E.g.
        - -2212 - This is an anti-proton 
- `MCParticles.generatorStatus`
    - This is an integer tag telling us what our underlying MC event generator tagged the particle as. This is less generic, but in this case, there are two very important tags
        - 4 - This is a **beam particle**. Our simulation is an electron-proton collision. Anything with status 4 is one of the initial beam particles.
        - 1 - This is a stable, final state particle. **This is something we would actually detect with out detector system in the collision.**

So, now to the actual *task* here. Using TTree->Draw() commands, do the following:

- Plot the Z-momentum distribution of **beam electrons**.
- Plot the Z-momentum distribution of **beam protons**.
- Assuming that the colliding beams are directly along the Z-axis, use the answers to the above to determine the collision energy of the events in the file you're looking at.
- Identify and plot the Z-momentum distribution of **final state electrons**.
- Create a 2D histogram plotting **beam electron Z-momentum** vs **final state electron Z-momentum**

In all cases, identify and use a suitable range for your axes and bins.

**Hint** - Carefully consider what properties each set of particles should have.

# Analysing via a Macro

So far, we have discussed drawing directly from the tree as a whole. But now let's pause and think about our tree a bit more.

Our tree has branches and collections that contain information on an event by event basis. In this case, each "event" is a simulated *ep* collision. We then have some output from that. When analysing, we typically want to sift through our events for ones of interest for one reason or another.

For example, maybe we only want events where we produce certain products or where events go in a certain angular range.

Most of the time, the analysis and sifting we want to do is beyond a single line TTree draw command. As such, we typically write macros to carry out a series of commands as we saw previously. But how can we sort through events in a tree? 

We'll explore two approaches today:

- TTreeReaders
    - This is the *classic* approach to doing ROOT analysis. Here, we read our tree, event by event, and apply cuts as we go to sort through what we do with the information in the event. For the majority of you, this will likely be how you will use ROOT.
- Dataframes
    - This is the *modern* approach to ROOT. The ROOT development team wants everyone to gradually go in this direction. If you're familiar with R or Python, this approach may seem more familiar. Ultimately, most analyses will go this direction, but the momentum behind this shift is only really starting. In nuclear physics, I suspect there will be some time delay before we start to see this more commonly

## TTree Readers

TTreeReader and the associated TTreeReaderArray is a simple interface for reading data from a TTree. The class description and examples can be seen in the [ROOT TTreeReader class documentation](https://root.cern/doc/v640/classTTreeReader.html). To instantiate the reader and access values from a given branch (e.g. the MCParticles branch), one would use the following calls:

In [ ]:
// Set up an input file chain
// We can add many files to the chain we wish to analyse
TChain *mychain = new TChain("events"); // Specify the tree we wish to analyse in each file.
mychain->Add("Day_1_Tutorial_Input.root");

// Initialise the reader
TTreeReader tree_reader(mychain);

Now, we need to tell the reader which branches/collections we actually want to look at:

In [ ]:
TTreeReaderArray<int> partGenStat(tree_reader, "MCParticles.generatorStatus"); // Our arguments are the reader we're using and the branch we want

Note that we actually need to know what type of thing we're reading from the tree - e.g. int, float etc. So far, all we've done is set up *what* we will access and read. We now need to use our reader to loop over the file:

In [ ]:
while(tree_reader.Next()) { // Loop over events
 for(unsigned int i=0; i<partGenStat.GetSize(); i++) // Loop through particles in the event.
   {
     int particleStatus = partGenStat[i]; // Access data-members as you would an array
     cout << particleStatus << endl;
   }
}

Note that we could add cuts and conditions within our loop. E.g.

In [ ]:
// Set up an input file chain
// We can add many files to the chain we wish to analyse
TChain *mychain = new TChain("events"); // Specify the tree we wish to analyse in each file.
mychain->Add("Day_1_Tutorial_Input.root");

// Initialise the reader
TTreeReader tree_reader(mychain);

TTreeReaderArray<int> partGenStat(tree_reader, "MCParticles.generatorStatus"); // Our arguments are the reader we're using and the branch we want

while(tree_reader.Next()) { // Loop over events
 for(unsigned int i=0; i<partGenStat.GetSize(); i++) // Loop through particles in the event.
   {
     int particleStatus = partGenStat[i]; // Access data-members as you would an array.
     if(particleStatus != 4) continue; // If the particle status is NOT 4, continue
     cout << particleStatus << endl;
   }
}

### Vectors

As you might guess from our previous task, some of the information in our ROOT file may be components of momentum. However, typically we don't want just one component, we want all of them. We want a **vector** which we can manipulate and get information from. 

ROOT has a series of vector classes with operations pre-defined. We can look at the [class definition of these vectors here](https://root.cern.ch/doc/v640/group__GenVector.html).

**Note**, these replace the deprecated TVector3 and TLorentzVector classes that you may encounter. For modern ROOT installs (6.30 onwards), use the TMath::Vector classes.

Given the information in our `MCParticles` branch, the most appropriate vector type is `ROOT::Math::PxPyPzMVector`. We have all components of the three momentum and the mass for our particles. We can create and fill a vector of this type and then plot some information from it.

In our loop above, as soon as we're at the level of *particles* **within** an event, we could start assigning components to a vector:

`ROOT::Math::PxPyPzMVector PartVec(partMomX[i], partMomY[i]...`

Assuming we have defined `partMomX` as a TTreeReaderArray as we did earlier for the `generatorStatus` for example.

### Task

So far, we have this just as cells on our notebook. Your task is the following:

- Create a macro which opens our file
  1. For every event, create a PxPyPzM vector for each **beam** and **final state** MCParticle in the event.
  2. Fill a histogram with the **energy** for all MC particles.
  3. Fill a separate histogram with the **energy** for all **final state electrons**.
  4. Create a 2D histogram of **energy** vs **theta** for all **final state electrons**.

**Hint** - `Vector.Theta()` will provide the polar angle in **radians**. Use `TMath::RadToDeg()` to convert from radians to degrees if you prefer them. Just multiply your value in radians by this quantity to get a value in degrees.

Print your output histograms to .pdfs.

#### Extension Task

Only do this if you have **also** done the task above using Dataframes.

A quantity we often want to know in collisions of this type is the $Q^{2}$, a measure of the 4-momentum transfer in the scattering interaction. $Q^{2}$ is given calculated as - 

$Q^{2} = -\left(p_{i} - p_{f}\right)^{2}$

where $p_{i}$ and $p_{f}$ are the initial and final 4-momentum of the electron in the interaction. In your macro:

- Calculate the truth value of $Q^{2}$ for each event
    - Create a histogram of $Q^{2}$ vs $\theta$ of **the final state electrons**
 
Save your histogram to a pdf as above.

Note, you may have more than one final state electron per event. You should calculate Q2 for each one. You may want to initialise your FS electron candidates as a vector, then add to that vector every time you find one:

`FS_Elec` in the final for loop are the individual vectors for the FS electrons we found. With this approach, we could calculate a quantity for every one.

**Thought** - What would happen if we did *not* do this?

## Dataframes

Dataframes are a more modern approach to analysing ROOT files. Going forward, TTreeReaders will gradually be phased out in newer versions of ROOT.

The intention is that everyone transitions to Dataframe based analyses over time. Part of the reason for this is skill transferability. A dataframe based approach is more similar to modern python or R techniques and approaches. We will look at how we can repeat our tasks from before using dataframes.

Another benefit of RDataFrames is that they can handle more modern C++ based techniques such as Lambda expressions and functions directly in filters (cuts).

The first step is to define our dataframe, i.e. tell it which file and tree to look at:

In [ ]:
ROOT::RDataFrame df("events", "Day_1_Tutorial_Input.root")

We can check what's in our dataframe (print the columns) with:

In [ ]:
cout << "Columns in the dataset: [ ";
for (const auto& colName : df.GetColumnNames()) {
    cout << colName << " ";
}
cout << "]" << endl;

We can also enable multithreading for faster processing with:

In [ ]:
//ROOT::EnableImplicitMT();

**However**, we will not do this (hence why it is commented out) as it will mess with our attempts to display the dataframe values later.

Now, we need to actually **do** something with our dataframe. We'll do this by defining some filters and applying them. We can also define histograms to be filled for conditions matching our filters.

To avoid repeating actions, we should try to follow this data-flow:

* Build a data-frame object by specifying your data-set
* Apply a series of transformations to your data.
  * Filters - E.g. applying some cuts
  * Define a new column - E.g. calculate some new quantity post filtering.
* Apply actions to the transformed data to produce results (e.g. fill a histogram).

We should **avoid** filling histograms until we've done all the transformations we want to do. What does a filter look like though?

Well, suppose we want to select only ``generatorStatus == 4`` events, we could do:

In [ ]:
auto df_BeamPartsFilter = df.Define("BeamPartsFilter", "MCParticles.generatorStatus == 4");

On its own, this doesn't really do much though. Effectively, what we have done is create an array of true/false values. Any particles in an event with `Status == 4` will be `true`. 

We need to apply this filter (sometimes called a mask) to build a new column of events that fulfil this condition. Let's try this by selecting out the PDG values for events with ``generatorStatus == 4`` and printing the first 10 events:

In [ ]:
auto df_Beam_PDG = df_BeamPartsFilter.Define("Beam_PDG", "MCParticles.PDG[BeamPartsFilter]");
auto my_display = df_Beam_PDG.Display({"Beam_PDG"}, 10);
my_display->Print();

We probably don't want to do just one thing at a time though. We could create a filter to select out our final beam electrons and retain their momentum values in one step for example. We could do this via:

In [ ]:
auto df_ElecExample = df
    // Define our selection filters
    .Define("FSElectrons", "MCParticles.generatorStatus == 1 && MCParticles.PDG == 11")
    // Define a columns of our double info
    .Define("FSElec_Px", "MCParticles.momentum.x[FSElectrons]")
    .Define("FSElec_Py", "MCParticles.momentum.y[FSElectrons]")
    .Define("FSElec_Pz", "MCParticles.momentum.z[FSElectrons]")
    .Define("FSElec_M",  "MCParticles.mass[FSElectrons]");
// Note - To get this to print nicely in our notebook we have to tell it to make this as a string and print it
auto electron_table = df_ElecExample.Display({"FSElec_Px", "FSElec_Py", "FSElec_Pz", "FSElec_M"}, 10)->AsString();
electron_table

We could also immediately create a vector from the quantities we've selected here rather than retaining the values themselves:

In [ ]:
auto df_ElecExample2 = df
    // Define our selection filters
    .Define("FSElectrons", "MCParticles.generatorStatus == 1 && MCParticles.PDG == 11")
    // Define a columns of our double info
    .Define("FSElec_4Vec", [](const ROOT::RVec<double>& x, 
								      const ROOT::RVec<double>& y, 
								      const ROOT::RVec<double>& z, 
								      const ROOT::RVec<double>& m,
								      const ROOT::RVec<int>& mask) {
    ROOT::RVec<ROOT::Math::PxPyPzMVector> vecs;
    vecs.reserve(x.size());
    for (size_t i = 0; i < x.size(); ++i) {
      if (mask[i]) {
	vecs.emplace_back(x[i], y[i], z[i], m[i]);
      }
    }
    return vecs;
  }, {"MCParticles.momentum.x", "MCParticles.momentum.y", "MCParticles.momentum.z", "MCParticles.mass", "FSElectrons"});

auto electron_table2 = df_ElecExample2.Display({"FSElec_4Vec"}, 10)->AsString();
electron_table2

Note, rather than tediously doing the above every time we need a vector, we could define a lambda helper that we can re-use instead:

In [ ]:
  // Reusable helper lambda to build 4-vectors from masked inputs
  auto make_4vecs = [](const ROOT::RVec<double>& x, 
                       const ROOT::RVec<double>& y, 
                       const ROOT::RVec<double>& z, 
                       const ROOT::RVec<double>& m,
                       const ROOT::RVec<int>& mask) {
    ROOT::RVec<ROOT::Math::PxPyPzMVector> vecs;
    vecs.reserve(x.size());
    for (size_t i = 0; i < x.size(); ++i) {
      if (mask[i]) {
        vecs.emplace_back(x[i], y[i], z[i], m[i]);
      }
    }
    return vecs;
  };

In [ ]:
auto df_ElecExample3 = df
    // Define our selection filters
    .Define("FSElectrons", "MCParticles.generatorStatus == 1 && MCParticles.PDG == 11")
    // Define a columns of our double info
    .Define("FSElec_4Vec", make_4vecs, {"MCParticles.momentum.x", "MCParticles.momentum.y", "MCParticles.momentum.z", "MCParticles.mass", "FSElectrons"});


auto electron_table3 = df_ElecExample3.Display({"FSElec_4Vec"}, 10)->AsString();
electron_table3

We now effectively have a vector of vectors, what if we want to extract information from our vectors? Well, we can add that as another column:

In [ ]:
auto df_ElecExample4 = df
    // Define our selection filters
    .Define("FSElectrons", "MCParticles.generatorStatus == 1 && MCParticles.PDG == 11")
    // Define a columns of our double info
    .Define("FSElec_4Vec", make_4vecs, {"MCParticles.momentum.x", "MCParticles.momentum.y", "MCParticles.momentum.z", "MCParticles.mass", "FSElectrons"})
    .Define("FSElec_E", [](const ROOT::RVec<ROOT::Math::PxPyPzMVector>& vecs) {
      ROOT::RVec<double> E; E.reserve(vecs.size());
      for (const auto& v : vecs) E.push_back(v.E());
      return E;
    }, {"FSElec_4Vec"});

auto electron_table4 = df_ElecExample4.Display({"FSElec_4Vec", "FSElec_E"}, 10)->AsString();
electron_table4

Similar to our vector function, we could make ones to get energies or other quantities if we'll be using them a lot:

In [ ]:
// Helpers to get the energy values from a vector
  auto get_E = [](const ROOT::RVec<ROOT::Math::PxPyPzMVector>& vecs) {
    return ROOT::VecOps::Map(vecs, [](const ROOT::Math::PxPyPzMVector& v) { return v.E(); });
  };

auto df_ElecExample5 = df
    // Define our selection filters
    .Define("FSElectrons", "MCParticles.generatorStatus == 1 && MCParticles.PDG == 11")
    // Define a columns of our double info
    .Define("FSElec_4Vec", make_4vecs, {"MCParticles.momentum.x", "MCParticles.momentum.y", "MCParticles.momentum.z", "MCParticles.mass", "FSElectrons"})
    .Define("FSElec_E",     get_E,        {"FSElec_4Vec"});


auto electron_table5 = df_ElecExample5.Display({"FSElec_4Vec", "FSElec_E"}, 10)->AsString();
electron_table5

We could also skip our function to define a vector column with filters entirely by just turning **all** particles into vectors, then applying a filter to that column to select the particles we want:

In [ ]:
auto df_ElecExample6 = df
    // Define our selection filters
    .Define("FSElectrons", "MCParticles.generatorStatus == 1 && MCParticles.PDG == 11")
    // Turn all particles in all events to vectors, then use our filters to select the subsets we want later
    .Define("Vecs", [](const ROOT::RVec<double>& x, const ROOT::RVec<double>& y,
		       const ROOT::RVec<double>& z, const ROOT::RVec<double>& m) {
      return ROOT::VecOps::Construct<ROOT::Math::PxPyPzMVector>(x, y, z, m);
    }, {"MCParticles.momentum.x", "MCParticles.momentum.y", "MCParticles.momentum.z", "MCParticles.mass"})
    .Define("FSElec_4Vec",   "Vecs[FSElectrons]")
    // Columns with calculated info from 4-vectors using our helper functions
    .Define("FSElec_E",     get_E,        {"FSElec_4Vec"});

auto electron_table6 = df_ElecExample6.Display({"FSElec_4Vec", "FSElec_E"}, 10)->AsString();
electron_table6

Depending upon how many vectors we want or how many different branches we're looking at, the *best* way to do this may vary a little.

Ok though, what about filling a histogram? Well, we could do this via:

In [ ]:
auto h1MCE_FSElec = df_ElecExample6.Histo1D(
				 {"h1MCE_FSElec", "E_{MC} for all FS Particles; E (GeV); Entries/0.2 GeV", 250, 0, 50}, 
				 "FSElec_E"
				 );

TCanvas* c_E_FSElec = new TCanvas("c_E_FSElec", "e_FS", 100, 0, 800, 600);
h1MCE_FSElec->Draw("HISTER");
c_E_FSElec->Draw();

**Remember** - You should apply any transformations and define any new columns (i.e. bits of info) you want **before** filling any histograms etc.

### Task

Now, repeat our TTree reader tasks but using RDataFrames instead. As a reminder, the tasks were:

- Create a macro which opens our file and makes a dataframe from the events tree.
  1. For every event, create a PxPyPzM vector for each **beam** and **final state** MCParticle in the event.
  2. Fill a histogram with the **energy** for all MC particles.
  3. Fill a separate histogram with the **energy** for all **final state electrons**.
  4. Create a 2D histogram of **energy** vs **theta** for all **final state electrons**.

Print your output histograms to .pdfs.

To get you started, open a file called `Session2_RDF.C` and paste the following in:

This does steps 1 (sort of) and 3 already. You will need to add the remaining histograms and calculations yourself.

#### Extension Task

In your macro:

- Calculate the truth value of $Q^{2}$ for each event
    - Create a histogram of $Q^{2}$ vs $\theta$ of **the final state electrons**

Save your histogram to a pdf as above.

As a reminder, 

$Q^{2} = -\left(p_{i} - p_{f}\right)^{2}$

where $p_{i}$ and $p_{f}$ are the initial and final 4-momentum of the electron in the interaction.